In [ ]:
import polars as pl
import numpy as np
import torch

# 读取训练数据
finance = pl.read_csv("example_data/finance_fixed.csv", schema_overrides={"股票代码": pl.Utf8})
train_data0 = pl.read_parquet("example_data/2025-10-21-ohlcv-30s.pq").sort(['Code', 'TimeInterval'])

In [ ]:
cap_sorted = finance.sort('总资产', descending=True)

# 分组
n = cap_sorted.height
n_huge = 100
n_large = int(n * 0.2) - n_huge
n_med = int(n * 0.4)
n_small = n - n_large - n_med

# 超大杯，大杯，中杯，小杯
cap_df = cap_sorted.with_columns(
    pl.int_range(pl.len()).alias('row_index')
).with_columns(
    pl.when(pl.col('row_index') < n_huge)
    .then(pl.lit('huge'))
    .when(pl.col('row_index') < n_huge + n_large)
    .then(pl.lit('large'))
    .when(pl.col('row_index') < n_huge + n_large + n_med)
    .then(pl.lit('med'))
    .otherwise(pl.lit('small'))
    .alias('cap_size')
).drop('row_index')

# 权重
cap_df = cap_df.with_columns(
    (pl.col('总资产') / pl.col('总资产').sum().over('cap_size')).alias('weight')
)

# clamp 超大杯权重不超过5%
huge_data = cap_df.filter(pl.col('cap_size') == 'huge')
huge_max_weight = 0.05
huge_total_clamped = huge_data.with_columns(
    pl.col('总资产').map_elements(lambda x: min(x, huge_max_weight * huge_data['总资产'].sum()), return_dtype=pl.Float64)
)['总资产'].sum()

# 应用 clamp 后的权重
cap_df = cap_df.with_columns(
    pl.when(pl.col('cap_size') == 'huge')
    .then(
        pl.col('总资产').map_elements(lambda x: min(x, huge_max_weight * huge_data['总资产'].sum()), return_dtype=pl.Float64) / huge_total_clamped
    )
    .otherwise(pl.col('weight'))
    .alias('weight')
)

# 仅保留需要的列
cap_df = cap_df.select(['股票代码', '总资产', 'cap_size', 'weight'])


In [ ]:
# 计算合成指数VWAP
train_data_with_vwap = train_data0.with_columns(
    (pl.col('Turnover') / pl.col('Volume')).alias('VWAP')
)

# 对齐股票代码
train_with_cap = train_data_with_vwap.join(
    cap_df.select(['股票代码', 'cap_size', 'weight']),
    left_on='Code',
    right_on='股票代码',
    how='left'
)

# 移除没计入指数股票
train_with_cap = train_with_cap.filter(pl.col('cap_size').is_not_null())

train_with_cap

In [ ]:
# 计算合成指数
cap_index = train_with_cap.group_by(['cap_size', 'TimeInterval']).agg([
    (pl.col('High') * pl.col('weight')).sum().alias('High'),
    (pl.col('Low') * pl.col('weight')).sum().alias('Low'),
    (pl.col('VWAP') * pl.col('weight')).sum().alias('VWAP')
]).select(['cap_size', 'TimeInterval', 'High', 'Low', 'VWAP']).sort(['cap_size', 'TimeInterval'])

# A股有涨跌幅限制，可以直接使用涨跌幅作为归一化
cap_index = cap_index.with_columns([
    (pl.col('High') / pl.col('High').first().over('cap_size') - 1).alias('High'),
    (pl.col('Low') / pl.col('Low').first().over('cap_size') - 1).alias('Low'),
    (pl.col('VWAP') / pl.col('VWAP').first().over('cap_size') -1).alias('VWAP')
])

# 移除第一行（全0行）
cap_index_filtered = cap_index.with_columns(
    pl.int_range(pl.len()).over('cap_size').alias('row_num')
).filter(
    pl.col('row_num') != 0
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 预览下数据哈

sns.set_style("whitegrid")
plt.figure(figsize=(14, 8))

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
cap_index_pd = cap_index_filtered.to_pandas()

# Plot High prices
sns.lineplot(data=cap_index_pd, x='row_num', y='High', hue='cap_size', ax=axes[0], linewidth=2)
axes[0].set_title('High Prices by Market Cap Size', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time Index')
axes[0].set_ylabel('Normalized High (% Change)')
axes[0].legend(title='Market Cap Size', loc='best')
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.3)

# Plot Low prices
sns.lineplot(data=cap_index_pd, x='row_num', y='Low', hue='cap_size', ax=axes[1], linewidth=2)
axes[1].set_title('Low Prices by Market Cap Size', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time Index')
axes[1].set_ylabel('Normalized Low (% Change)')
axes[1].legend(title='Market Cap Size', loc='best')
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)

# Plot VWAP
sns.lineplot(data=cap_index_pd, x='row_num', y='VWAP', hue='cap_size', ax=axes[2], linewidth=2)
axes[2].set_title('VWAP by Market Cap Size', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Time Index')
axes[2].set_ylabel('Normalized VWAP (% Change)')
axes[2].legend(title='Market Cap Size', loc='best')
axes[2].axhline(y=0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 计算股票涨跌幅作为归一化训练数据

train_data = train_data_with_vwap.select(['Code', 'TimeInterval', 'High', 'Low', 'VWAP']).with_columns(
    (pl.col('High') / pl.col('High').first().over('Code') - 1).alias('High'),
    (pl.col('Low') / pl.col('Low').first().over('Code') - 1).alias('Low'),
    (pl.col('VWAP') / pl.col('VWAP').first().over('Code') -1).alias('VWAP')
).with_columns(
    pl.int_range(pl.len()).over('Code').alias('row_num')
).filter(
    pl.col('row_num') != 0
)

# 股价有强烈时间相关性，加入周期特征到训练集

train_data = train_data.with_columns(
    (2 * np.pi * (pl.col('row_num') / pl.col('row_num').max().over('Code'))).alias('phase')
).with_columns(
    np.sin(pl.col('phase')).alias('phase_sin'),
    np.cos(pl.col('phase')).alias('phase_cos')
)

# 合成指数并入训练数据

for cap_size in ['huge', 'large', 'med', 'small']:
    cap_subset = cap_index_filtered.filter(pl.col('cap_size') == cap_size).select([
        'TimeInterval',
        pl.col('VWAP').alias(f'VWAP_{cap_size}')
    ])

    train_data = train_data.join(
        cap_subset,
        on='TimeInterval',
        how='left'
    )

In [ ]:
# 特征集

feature_columns = [
    'High', 'Low', 'VWAP', 'phase_sin',
    'VWAP_huge','VWAP_large','VWAP_med','VWAP_small'
]

# 设定窗口

lookback_horizon = 40
forecast_horizon = 10
# 窗口有重叠，滚动增加特化性
overlap_horizon = 30

print("使劲准备数据")

# only select Code: '000001.SZ' for testing
train_set = train_data.filter(pl.col('Code') == '000001')
grouped = train_set.group_by('Code', maintain_order=True)

X_list = []
Y_list = []
for code, group_df in grouped:
    features = group_df.select(feature_columns).to_numpy()

    # 目标特征股价VWAP
    target = features[:, 2]  # VWAP column

    n_samples = len(features)

    for i in range(0, n_samples - lookback_horizon - forecast_horizon + 1, lookback_horizon - overlap_horizon):
        # 输入窗口
        X_window = features[i:i + lookback_horizon]

        # 目标窗口
        Y_window = target[i + lookback_horizon:i + lookback_horizon + forecast_horizon]

        # Only add if we have complete windows
        if len(X_window) == lookback_horizon and len(Y_window) == forecast_horizon:
            X_list.append(X_window)
            Y_list.append(Y_window)

train_X = np.array(X_list, dtype=np.float32)  # Shape: (num_samples, lookback_horizon, num_features)
train_Y = np.array(Y_list, dtype=np.float32)  # Shape: (num_samples, forecast_horizon)

print(f"train_X shape: {train_X.shape}")
print(f"train_Y shape: {train_Y.shape}")
print(f"Number of training samples: {len(train_X)}")
print(f"\nSample statistics:")
print(f"train_X - min: {train_X.min():.6f}, max: {train_X.max():.6f}, mean: {train_X.mean():.6f}")
print(f"train_Y - min: {train_Y.min():.6f}, max: {train_Y.max():.6f}, mean: {train_Y.mean():.6f}")

In [ ]:
from model.model_config import ModelConfig
from model.overthink import OverthinkModel

# 设置模型参数
config = ModelConfig(
  feature_num=len(feature_columns),
  lookback_horizon=lookback_horizon,
  forecast_horizon=forecast_horizon,
  batch_size=16,
  high_freq_step=6,
  low_freq_step=3,
  hidden_layer_num=2,
  hidden_size=512,
  head_num=8,
  use_causal=True,
  use_rope=True,
  expansion_factor=4.0,
  forecast_aggregation='mean',
  forecast_ema_period=5,
)

# Initialize model
print("\n2. Initializing model...")
model = OverthinkModel(config)
num_params = sum(p.numel() for p in model.parameters())
print(f"   - Total parameters: {num_params:,}")

# Determine device and move model
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"   - Using CUDA device: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("   - Using MPS device (Apple Silicon)")
else:
    device = torch.device("cpu")
    print("   - Using CPU device")

model = model.to(device)
print(f"   - Model moved to {device}")

In [ ]:
from block.trend_loss import MultiScaleTrendDirectionLoss
from torch.utils.data import TensorDataset, DataLoader

# 腾挪数据到设备
train_X_tensor = torch.from_numpy(train_X).to(device)
train_Y_tensor = torch.from_numpy(train_Y).to(device)

dataset = TensorDataset(train_X_tensor, train_Y_tensor)
dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True)

calc_alpha = lambda n : 2.0 / (n + 1)
a1 = calc_alpha(6) # 计算6期EMA的alpha (30s x 6 = 3分钟)
a2 = calc_alpha(15) # 计算15期EMA的alpha (30s x 15 = 7.5分钟)
a3 = calc_alpha(30) # 计算30期EMA的alpha (30s x 30 = 15分钟)

# 方向增强损失
criterion = MultiScaleTrendDirectionLoss(
    alphas=[a1, a2, a3],  # 短，中，长期趋势
    weights=[0.75, 1.0, 0.75],  # 稍微关注中期趋势
    reduction="mean",
    loss_type="softsign",
    huber_margin=0.0,  # Not used for softsign
    softness=0.1,
    norm_eps=1e-8
)

# 优化器
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# LR规划器
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

print("Training setup complete!")
print(f"Dataset size: {len(dataset)}")
print(f"Batches per epoch: {len(dataloader)}")
print(f"Loss function: MultiScaleTrendDirectionLoss (softsign, softness=0.1)")
print(f"Optimizer: AdamW (lr=1e-4, weight_decay=1e-5)")

In [ ]:
# ai生成的训练代码

import time

num_epochs = 100
train_losses = []
best_loss = float('inf')
best_epoch = 0

print("\n" + "="*70)
print(" "*20 + "🚀 TRAINING START 🚀")
print("="*70)
print(f"📊 Total Epochs: {num_epochs}")
print(f"🎯 Batch Size: {config.batch_size}")
print(f"📦 Total Batches per Epoch: {len(dataloader)}")
print(f"🔢 Total Training Samples: {len(dataset)}")
print("="*70 + "\n")

start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    epoch_start = time.time()

    for batch_idx, (X_batch, Y_batch) in enumerate(dataloader):
        # Forward pass
        predictions = model(X_batch)  # Shape: [batch_size, forecast_horizon, feature_num]

        # Extract VWAP predictions only (index 2 in features)
        vwap_predictions = predictions[:, :, 2]  # Shape: [batch_size, forecast_horizon]

        # Compute loss
        loss = criterion(vwap_predictions, Y_batch)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        epoch_loss += loss.item()

    # Update learning rate
    scheduler.step()

    # Calculate metrics
    avg_loss = epoch_loss / len(dataloader)
    train_losses.append(avg_loss)
    epoch_time = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]['lr']

    # Track best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_epoch = epoch + 1
        best_indicator = "⭐ NEW BEST"
    else:
        best_indicator = ""

    # Print progress with enhanced formatting
    if (epoch + 1) % 5 == 0 or epoch == 0:
        elapsed = time.time() - start_time
        eta = (elapsed / (epoch + 1)) * (num_epochs - epoch - 1)

        print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
              f"Loss: {avg_loss:.6f} | "
              f"LR: {current_lr:.2e} | "
              f"Time: {epoch_time:.2f}s | "
              f"ETA: {eta/60:.1f}min {best_indicator}")

total_time = time.time() - start_time

print("\n" + "="*70)
print(" "*20 + "✅ TRAINING COMPLETE ✅")
print("="*70)
print(f"⏱️  Total Training Time: {total_time/60:.2f} minutes")
print(f"📉 Final Loss: {train_losses[-1]:.6f}")
print(f"🏆 Best Loss: {best_loss:.6f} (Epoch {best_epoch})")
print(f"📈 Improvement: {((train_losses[0] - best_loss) / train_losses[0] * 100):.2f}%")
print("="*70)

In [ ]:
# Plot training metrics with enhanced visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Full training loss
axes[0, 0].plot(train_losses, linewidth=2.5, color='#2E86AB', alpha=0.8)
axes[0, 0].axhline(y=best_loss, color='red', linestyle='--', alpha=0.5, label=f'Best: {best_loss:.6f}')
axes[0, 0].scatter([best_epoch-1], [best_loss], color='red', s=100, zorder=5, marker='★')
axes[0, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_title('📉 Training Loss - Full History', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, linestyle='--')
axes[0, 0].legend(loc='upper right')

# Plot 2: Loss after warmup (epoch 10+)
if len(train_losses) > 10:
    axes[0, 1].plot(range(10, len(train_losses)), train_losses[10:],
                    linewidth=2.5, color='#A23B72', alpha=0.8)
    axes[0, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[0, 1].set_ylabel('Loss', fontsize=12, fontweight='bold')
    axes[0, 1].set_title('📊 Training Loss - After Warmup (Epoch 10+)', fontsize=14, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, linestyle='--')

# Plot 3: Loss improvement rate (derivative)
if len(train_losses) > 1:
    loss_diff = np.diff(train_losses)
    axes[1, 0].plot(loss_diff, linewidth=2, color='#F18F01', alpha=0.7)
    axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[1, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Loss Change', fontsize=12, fontweight='bold')
    axes[1, 0].set_title('📈 Loss Improvement Rate (Δ Loss)', fontsize=14, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3, linestyle='--')
    axes[1, 0].fill_between(range(len(loss_diff)), loss_diff, 0,
                            where=(np.array(loss_diff) < 0), alpha=0.3, color='green', label='Improving')
    axes[1, 0].fill_between(range(len(loss_diff)), loss_diff, 0,
                            where=(np.array(loss_diff) >= 0), alpha=0.3, color='red', label='Degrading')
    axes[1, 0].legend(loc='upper right')

# Plot 4: Training statistics summary
axes[1, 1].axis('off')
stats_text = f"""
📊 TRAINING SUMMARY
{'='*40}

🏆 Best Loss:        {best_loss:.6f}
📍 Best Epoch:       {best_epoch}
📉 Final Loss:       {train_losses[-1]:.6f}
📈 Initial Loss:     {train_losses[0]:.6f}

💪 Total Improvement: {((train_losses[0] - best_loss) / train_losses[0] * 100):.2f}%
📊 Loss Reduction:    {train_losses[0] - best_loss:.6f}

🔢 Total Epochs:     {len(train_losses)}
📦 Batch Size:       {config.batch_size}
🎯 Samples:          {len(dataset)}

⚙️  Model Parameters: {num_params:,}
🧠 Hidden Size:      {config.hidden_size}
👁️  Attention Heads:  {config.head_num}
"""
axes[1, 1].text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
                verticalalignment='center', bbox=dict(boxstyle='round',
                facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.show()

# Print summary
print("\n" + "="*70)
print(" "*25 + "📊 TRAINING SUMMARY")
print("="*70)
print(f"🏆 Best Loss: {best_loss:.6f} at epoch {best_epoch}")
print(f"📉 Final Loss: {train_losses[-1]:.6f}")
print(f"💪 Total Improvement: {((train_losses[0] - best_loss) / train_losses[0] * 100):.2f}%")
print("="*70)